## Example 1: Encode one event-stream message and decode it back

In [ ]:
import importlib.metadata
import io

from smithy_aws_event_stream.events import Event, EventMessage

# Example 1: the binary frame Bedrock and Transcribe stream: prelude, headers, payload and two CRC32s
assert importlib.metadata.version("smithy-aws-event-stream") == "0.3.0"
message = EventMessage(headers={":message-type": "event", ":event-type": "chunk", ":content-type": "application/json"}, payload=b'{"bytes":"aGk="}')
raw = message.encode()
event = Event.decode(io.BytesIO(raw))
assert event.message.headers == message.headers and event.message.payload == message.payload
assert event.prelude.total_length == len(raw)
print("Example 1 OK:", len(raw), "bytes", event.message.headers[":event-type"])

## Example 2: Typed header values

In [ ]:
import datetime
import io
import uuid

from smithy_aws_event_stream.events import Event, EventMessage

# Example 2: headers carry booleans, integers, bytes, strings, timestamps and UUIDs
when = datetime.datetime(2026, 11, 7, 1, 2, 3, tzinfo=datetime.timezone.utc)
headers = {"flag": True, "count": 7, "blob": b"\x01\x02", "name": "x", "when": when, "id": uuid.UUID(int=5)}
back = Event.decode(io.BytesIO(EventMessage(headers=headers, payload=b"").encode())).message.headers
assert back["flag"] is True and back["count"] == 7 and back["blob"] == b"\x01\x02" and back["name"] == "x", back
assert back["when"] == when and back["id"] == uuid.UUID(int=5)
print("Example 2 OK:", sorted(back))

## Example 3: A corrupted frame is rejected by its checksum

In [ ]:
import io

from smithy_aws_event_stream.events import ChecksumMismatch, Event, EventMessage

# Example 3: flipping one payload byte breaks the message CRC32
raw = bytearray(EventMessage(headers={":event-type": "chunk"}, payload=b"hello").encode())
raw[-6] ^= 0xFF
try:
    Event.decode(io.BytesIO(bytes(raw)))
except ChecksumMismatch as error:
    print("Example 3 OK:", type(error).__name__)
else:
    raise AssertionError("a corrupted frame decoded")

## Example 4: Several frames in one stream

In [ ]:
import io

from smithy_aws_event_stream.events import Event, EventMessage

# Example 4: frames are self-delimiting, so a stream decodes one message after another
stream = io.BytesIO(b"".join(EventMessage(headers={":event-type": "chunk", "n": i}, payload=f"part{i}".encode()).encode() for i in range(3)))
payloads = []
while stream.tell() < len(stream.getbuffer()):
    payloads.append(Event.decode(stream).message.payload)
assert payloads == [b"part0", b"part1", b"part2"], payloads
print("Example 4 OK:", payloads)

## Example 5: The async decoder reads from an async byte stream

In [ ]:
import asyncio

from smithy_aws_event_stream.events import Event, EventMessage

# Example 5: decode_async reads a frame from an asyncio stream, the way a client reads a live response
raw = EventMessage(headers={":event-type": "chunk"}, payload=b"async!").encode()


class Source:
    def __init__(self, data):
        self.data = data
    async def read(self, size=-1):
        out, self.data = (self.data, b"") if size < 0 else (self.data[:size], self.data[size:])
        return out


event = await Event.decode_async(Source(raw))
assert event.message.payload == b"async!"
print("Example 5 OK:", event.message.payload)